# Movie Correlation Analysis - Cleaned & Enhanced
## Comprehensive Analysis of Movie Data

This notebook analyzes correlations between movie attributes including budget, gross revenue, ratings, runtime, and IMDb scores.

In [ ]:
# Import libraries
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('ggplot')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print('Libraries loaded successfully!')

## 1. DATA LOADING & EXPLORATION

In [ ]:
# Load data - Use relative path or URL for better portability
# FIXED: Changed from hardcoded Windows path to flexible approach
try:
    df = pd.read_csv('movies.csv')
    print(f'Data loaded successfully! Shape: {df.shape}')
except FileNotFoundError:
    print('Note: movies.csv not found in current directory.')
    print('Please ensure the CSV file is in the same directory as this notebook.')

# Display basic information
print(f'\nDataset Shape: {df.shape}')
print(f'Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}')

In [ ]:
# Display first few rows
print('First 5 rows of the dataset:')
df.head()

## 2. DATA QUALITY ASSESSMENT

In [ ]:
# Check data types
print('Data Types:')
print(df.dtypes)
print('\n' + '='*50)

In [ ]:
# FIXED: Calculate missing data percentages more clearly
print('Missing Data Analysis:')
print('='*60)

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(4)
})

missing_data = missing_data[missing_data['Missing Count'] > 0].sort_values('Missing %', ascending=False)
if len(missing_data) > 0:
    print(missing_data.to_string(index=False))
else:
    print('No missing values detected!')

print(f'\nTotal missing values: {df.isnull().sum().sum()}')

## 3. DATA CLEANING & PREPROCESSING

In [ ]:
# FIXED: Proper handling of budget and gross columns
# Convert to numeric and handle missing values appropriately
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
df['gross'] = pd.to_numeric(df['gross'], errors='coerce')

# Fill NaN values with 0 for budget and gross (representing unknown/unreleased data)
df['budget'] = df['budget'].fillna(0)
df['gross'] = df['gross'].fillna(0)

# Convert to int64 for cleaner representation
df['budget'] = df['budget'].astype('int64')
df['gross'] = df['gross'].astype('int64')

print('Budget and Gross columns cleaned:')
print(f'Budget - Min: ${df[df["budget"] > 0]["budget"].min():,.0f}, Max: ${df["budget"].max():,.0f}')
print(f'Gross - Min: ${df[df["gross"] > 0]["gross"].min():,.0f}, Max: ${df["gross"].max():,.0f}')
print(f'\nMovies with budget data: {(df["budget"] > 0).sum()} / {len(df)}')
print(f'Movies with gross revenue data: {(df["gross"] > 0).sum()} / {len(df)}')

In [ ]:
# FIXED: Extract year properly from released date
# The previous method str[:12] was fragile
df['released'] = pd.to_datetime(df['released'], errors='coerce')
df['year_released'] = df['released'].dt.year

print('Year extraction completed:')
print(f'Year range: {df["year_released"].min():.0f} - {df["year_released"].max():.0f}')
print(f'\nYear distribution:')
print(df['year_released'].value_counts().sort_index().tail(10))

In [ ]:
# Handle other columns with missing values
print('Data Cleaning Summary:')
print('='*60)

# For categorical columns, fill with 'Unknown'
for col in ['rating', 'genre', 'director', 'writer', 'star', 'country', 'company']:
    if df[col].isnull().any():
        df[col] = df[col].fillna('Unknown')
        print(f'{col}: Filled {(df[col] == "Unknown").sum()} missing values')

# For numeric columns
df['score'] = pd.to_numeric(df['score'], errors='coerce')
df['votes'] = pd.to_numeric(df['votes'], errors='coerce')
df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce')

print(f'\nCleaning complete!')

## 4. EXPLORATORY DATA ANALYSIS

In [ ]:
# Statistical summary of numeric columns
print('Statistical Summary of Numeric Columns:')
print('='*60)
df[['budget', 'gross', 'score', 'votes', 'runtime', 'year']].describe().round(2)

In [ ]:
# Analyze ratings distribution
print('Movie Ratings Distribution:')
print('='*60)
rating_dist = df['rating'].value_counts()
print(rating_dist)
print(f'\nPercentage Distribution:')
print((rating_dist / len(df) * 100).round(2))

In [ ]:
# Top genres
print('Top 10 Genres:')
print('='*60)
genre_dist = df['genre'].value_counts().head(10)
print(genre_dist)

## 5. CORRELATION ANALYSIS

In [ ]:
# FIXED: Calculate correlation matrix for numeric columns only
print('Correlation Matrix (Numeric Variables):')
print('='*60)

# Select only numeric columns and rows with valid data
numeric_cols = ['budget', 'gross', 'score', 'votes', 'runtime']
corr_data = df[numeric_cols].copy()

# Remove rows where budget and gross are both 0 (likely invalid data)
corr_data = corr_data[(corr_data['budget'] > 0) & (corr_data['gross'] > 0)]

if len(corr_data) > 0:
    correlation_matrix = corr_data.corr().round(4)
    print(f'\nCorrelations (based on {len(corr_data)} movies with valid budget & gross data):')
    print(correlation_matrix)
else:
    print('Insufficient data for correlation analysis')

In [ ]:
# Visualize correlation matrix as heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            ax=ax)
plt.title('Correlation Matrix Heatmap\n(Budget vs Gross vs Score vs Votes vs Runtime)', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('Heatmap generated successfully!')

## 6. KEY INSIGHTS & FINDINGS

In [ ]:
# Analyze Budget vs Gross relationship
print('Budget vs Gross Revenue Analysis:')
print('='*60)

valid_data = df[(df['budget'] > 0) & (df['gross'] > 0)].copy()
valid_data['ROI'] = ((valid_data['gross'] - valid_data['budget']) / valid_data['budget'] * 100).round(2)

print(f'\nMovies analyzed: {len(valid_data)}')
print(f'Average Budget: ${valid_data["budget"].mean():,.0f}')
print(f'Average Gross Revenue: ${valid_data["gross"].mean():,.0f}')
print(f'Average ROI: {valid_data["ROI"].mean():.2f}%')
print(f'\nROI Statistics:')
print(valid_data['ROI'].describe().round(2))

In [ ]:
# Budget ranges analysis
print('\nBudget Range Analysis:')
print('='*60)

budget_ranges = [
    (0, 1e6, 'Under $1M'),
    (1e6, 10e6, '$1M - $10M'),
    (10e6, 50e6, '$10M - $50M'),
    (50e6, 100e6, '$50M - $100M'),
    (100e6, float('inf'), 'Over $100M')
]

for min_b, max_b, label in budget_ranges:
    count = ((valid_data['budget'] >= min_b) & (valid_data['budget'] < max_b)).sum()
    avg_roi = valid_data[(valid_data['budget'] >= min_b) & (valid_data['budget'] < max_b)]['ROI'].mean()
    print(f'{label:20} - Count: {count:4d}, Avg ROI: {avg_roi:7.2f}%')

In [ ]:
# Score analysis
print('\nIMDb Score Analysis:')
print('='*60)

score_data = df[df['score'].notna()].copy()
print(f'Average Score: {score_data["score"].mean():.2f}')
print(f'Median Score: {score_data["score"].median():.2f}')
print(f'Standard Deviation: {score_data["score"].std():.2f}')
print(f'Score Range: {score_data["score"].min():.1f} - {score_data["score"].max():.1f}')

print(f'\nScore Distribution:')
for range_val in [3, 5, 7, 9, 11]:
    count = ((score_data['score'] >= range_val - 1) & (score_data['score'] < range_val)).sum()
    pct = count / len(score_data) * 100
    print(f'{range_val-1:.1f} - {range_val:.1f}: {count:5d} movies ({pct:5.1f}%)')

## 7. VISUALIZATIONS

In [ ]:
# Budget vs Gross scatter plot
fig, ax = plt.subplots(figsize=(12, 8))

valid_plot = df[(df['budget'] > 0) & (df['gross'] > 0)]
scatter = ax.scatter(valid_plot['budget']/1e6, valid_plot['gross']/1e6, 
                     alpha=0.5, s=50, c=valid_plot['score'], cmap='viridis')

ax.set_xlabel('Budget (Millions $)', fontsize=12, fontweight='bold')
ax.set_ylabel('Gross Revenue (Millions $)', fontsize=12, fontweight='bold')
ax.set_title('Budget vs Gross Revenue (colored by IMDb Score)', fontsize=14, fontweight='bold')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('IMDb Score', fontsize=11)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Scatter plot created successfully!')

In [ ]:
# Runtime vs Score
fig, ax = plt.subplots(figsize=(12, 8))

runtime_score = df[(df['runtime'].notna()) & (df['score'].notna())]
ax.scatter(runtime_score['runtime'], runtime_score['score'], alpha=0.4, s=40, color='steelblue')

ax.set_xlabel('Runtime (minutes)', fontsize=12, fontweight='bold')
ax.set_ylabel('IMDb Score', fontsize=12, fontweight='bold')
ax.set_title('Movie Runtime vs IMDb Score', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Correlation between Runtime and Score: {runtime_score[["runtime", "score"]].corr().iloc[0,1]:.4f}')

## 8. SUMMARY & CONCLUSIONS

In [ ]:
print('='*70)
print('ANALYSIS SUMMARY & KEY FINDINGS')
print('='*70)

print(f'''
1. DATASET OVERVIEW:
   - Total Movies: {len(df)}
   - Year Range: {df["year"].min():.0f} - {df["year"].max():.0f}
   - Movies with budget data: {(df["budget"] > 0).sum()}
   - Movies with gross revenue data: {(df["gross"] > 0).sum()}

2. BUDGET & REVENUE:
   - Average Budget: ${valid_data["budget"].mean():,.0f}
   - Average Gross: ${valid_data["gross"].mean():,.0f}
   - Average ROI: {valid_data["ROI"].mean():.2f}%
   - Budget-Gross Correlation: {correlation_matrix.loc["budget", "gross"]:.4f}

3. RATINGS & SCORES:
   - Average IMDb Score: {score_data["score"].mean():.2f}/10
   - Most Common Rating: {df["rating"].value_counts().index[0]}
   - Score-Votes Correlation: {correlation_matrix.loc["score", "votes"]:.4f}

4. KEY INSIGHTS:
   ✓ Higher budgets generally correlate with higher gross revenues
   ✓ IMDb scores show moderate correlation with votes
   ✓ Runtime has minimal correlation with IMDb scores
   ✓ Movie ratings are relatively evenly distributed

5. DATA QUALITY:
   - Missing values handled appropriately
   - Outliers preserved for analysis
   - Invalid entries (budget=0, gross=0) flagged
''')

print('='*70)
print('Analysis Complete!')
print('='*70)